# GF(8) generalized toric codes: evolutionary search on Kaggle

This notebook runs the fixed-dimension search from `evolve_search.py` on a Kaggle NVIDIA GPU. It is a candidate factory: results are sampled and marked `verified: false`; promising candidates still need a stronger recheck or Magma/BZ verification.

Kaggle setup:

1. Create a Kaggle Notebook.
2. Set **Accelerator** to a GPU such as T4, P100, or A100.
3. Attach this repository as a Kaggle Dataset, or copy the repo into `/kaggle/working/generalizedsearch`.
4. Run the cells below. JSONL result files are written to `/kaggle/working` and can be downloaded from the notebook output.

Note: this repository is currently GF(8), length 49. The often-mentioned `[36,19,12]` example is GF(7), length 36, and would need a separate GF(7) evaluation matrix and kernels.

In [ ]:
# Locate the repo files. This works when the repo is copied to /kaggle/working
# or attached as a Kaggle Dataset under /kaggle/input.
from pathlib import Path
import os
import sys

os.chdir("/kaggle/working")

candidate_roots = [
    Path("/kaggle/working/generalizedsearch"),
    Path("/kaggle/working/generalized-toric-gpu-search"),
    Path("/kaggle/working"),
]
candidate_roots.extend(p.parent for p in Path("/kaggle/input").glob("**/evolve_search.py"))

REPO = None
for root in candidate_roots:
    if (root / "evolve_search.py").exists() and (root / "kernel_cuda_bp.py").exists():
        REPO = root
        break

if REPO is None:
    raise FileNotFoundError(
        "Could not find evolve_search.py. Attach the repo as a Kaggle Dataset "
        "or copy it to /kaggle/working/generalizedsearch."
    )

sys.path.insert(0, str(REPO))
print(f"Using repo: {REPO}")

In [ ]:
# GPU sanity check. CuPy is normally preinstalled on Kaggle GPU images.
REQUIRE_CUDA = True

try:
    import cupy as cp
    n_gpu = cp.cuda.runtime.getDeviceCount()
    print(f"CuPy {cp.__version__}; CUDA devices: {n_gpu}")
    for i in range(n_gpu):
        props = cp.cuda.runtime.getDeviceProperties(i)
        print(
            f"Device {i}: {props['name'].decode()} "
            f"({props.get('multiProcessorCount', 0)} SMs, "
            f"{props['totalGlobalMem'] // 1024**3} GB VRAM)"
        )
except Exception as exc:
    if REQUIRE_CUDA:
        raise RuntimeError("Enable a Kaggle GPU accelerator before running the search.") from exc
    print(f"CUDA unavailable; evolve_search.py will fall back if possible: {exc}")

In [ ]:
# Choose the search target and runtime profile.
from codetables import bounds_for_n

TARGET_K = 19

try:
    TARGET_D = bounds_for_n(49, q=8, cache=True)[TARGET_K]
except Exception:
    # Offline fallback for common GF(8), n=49 targets.
    TARGET_D = {12: 28, 18: 21, 19: 21, 22: 18, 38: 7}.get(TARGET_K)
    if TARGET_D is None:
        raise

# Start with quick to test that the notebook and GPU are healthy.
# Switch to serious for an overnight-style run.
PRESET = "quick"  # "quick" or "serious"

if PRESET == "quick":
    GENERATIONS = 20
    POPULATION_SIZE = 300
    SAMPLE_COUNT = 50_000
    RECHECK_SAMPLE_COUNT = 500_000
    RECHECK_ROUNDS = 2
elif PRESET == "serious":
    GENERATIONS = 200
    POPULATION_SIZE = 300
    SAMPLE_COUNT = 200_000
    RECHECK_SAMPLE_COUNT = 2_000_000
    RECHECK_ROUNDS = 3
else:
    raise ValueError("PRESET must be 'quick' or 'serious'")

ELITE_COUNT = 30
PARENT_POOL = 200
MUTATION_RATE = 0.10
RECHECK_TOP = 8
RECHECK_EVERY = 10
SEED = 1
BATCH_SIZE = None  # None lets evolve_search choose from the GPU SM count.
SEED_SETS = []     # Optional: tuples/lists of 0..48 indices, each of length TARGET_K.

print(f"Target: GF(8) [49,{TARGET_K},?>={TARGET_D}]  preset={PRESET}")

In [ ]:
# Run the search. Candidates are written as JSONL records in /kaggle/working.
from datetime import datetime
from pathlib import Path
from evolve_search import run_evolution

ts = datetime.now().strftime("%Y%m%d_%H%M%S")
RESULTS_FILE = Path(f"/kaggle/working/evolve_k{TARGET_K}_{PRESET}_{ts}.json")

run_evolution(
    k=TARGET_K,
    target_d=TARGET_D,
    generations=GENERATIONS,
    population_size=POPULATION_SIZE,
    elite_count=ELITE_COUNT,
    parent_pool=PARENT_POOL,
    mutation_rate=MUTATION_RATE,
    sample_count=SAMPLE_COUNT,
    recheck_sample_count=RECHECK_SAMPLE_COUNT,
    recheck_rounds=RECHECK_ROUNDS,
    recheck_top=RECHECK_TOP,
    recheck_every=RECHECK_EVERY,
    batch_size=BATCH_SIZE,
    seed=SEED,
    seed_sets=[tuple(s) for s in SEED_SETS],
    results_file=RESULTS_FILE,
)

print(f"Done: {RESULTS_FILE}")

In [ ]:
# Inspect and download the result file.
import json
from IPython.display import FileLink, display

records = []
with open(RESULTS_FILE) as f:
    for line in f:
        line = line.strip()
        if line:
            records.append(json.loads(line))

candidates = [r for r in records if r.get("type") == "candidate"]
print(f"Records: {len(records)}; candidates: {len(candidates)}")

for rec in candidates[-10:]:
    print(
        rec.get("name"),
        "generation=", rec.get("generation"),
        "sampled=", rec.get("sampled_min_distance"),
        "recheck=", rec.get("recheck_distances"),
        "indices=", rec.get("indices"),
    )

display(FileLink(str(RESULTS_FILE)))

## What to do with candidates

The sampled search is deliberately optimistic: it tries to avoid spending BZ-level effort on obviously poor sets, but it does not prove the minimum distance. For finalists, increase `RECHECK_SAMPLE_COUNT` and `RECHECK_ROUNDS`, then export the best sets for Magma's `VerifyMinimumDistanceLowerBound` / `MinimumDistance` workflow.

For a serious Kaggle run, use `PRESET = "serious"`, vary `SEED`, and keep each output JSONL file. The result records contain raw `indices` and `lattice_points`; `indices` are the compact `a * 7 + b` encoding used by the repo.